# Databricks Connect Demo

This notebook demonstrates how to use Databricks Connect to run Spark code locally against a Databricks workspace.

**Configuration:**
- Profile: `DEFAULT`
- Workspace: e2-demo-field-eng.cloud.databricks.com
- Compute: Serverless

## 1. Connect to Databricks

In [2]:
from databricks.connect import DatabricksSession

# Create a Spark session connected to Databricks
spark = DatabricksSession.builder.profile("DEFAULT").getOrCreate()

print(f"Connected to Databricks!")
print(f"Spark version: {spark.version}")

Connected to Databricks!
Spark version: 4.1.0


## 2. List Available Catalogs

In [3]:
# List all catalogs in Unity Catalog
catalogs = spark.sql("SHOW CATALOGS")
catalogs.show(truncate=False)

+-------------------------------------------------------------------------------+
|catalog                                                                        |
+-------------------------------------------------------------------------------+
|00vsdb                                                                         |
|00vsdb2                                                                        |
|00vsdb3                                                                        |
|03-scp-glue                                                                    |
|10_16_share                                                                    |
|118553_catalog                                                                 |
|118_group_sample_uk_business_data_full_uk_b2b_coverage_marketable_or_analytical|
|202509_ski_poc                                                                 |
|2026_2_3_demo_nobunagasambition                                                |
|365scores      

## 3. Explore a Catalog

In [5]:
# Change this to a catalog you have access to
CATALOG = "sv_aibuilder_saas_demo"

# List schemas in the catalog
schemas = spark.sql(f"SHOW SCHEMAS IN {CATALOG}")
schemas.show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|saas_sandbox      |
|silver            |
+------------------+



In [6]:
# List tables in a schema
SCHEMA = "gold"

tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}")
tables.show(truncate=False)

+--------+-------------------------+-----------+
|database|tableName                |isTemporary|
+--------+-------------------------+-----------+
|gold    |agg_daily_product_metrics|false      |
|gold    |agg_user_lifetime_summary|false      |
|gold    |dim_user                 |false      |
|gold    |fact_event               |false      |
|gold    |fact_order               |false      |
+--------+-------------------------+-----------+



## 4. Query Sample Data

In [ ]:
# Query the NYC taxi trips table
df = spark.sql("""
    SELECT 
        pickup_datetime,
        dropoff_datetime,
        trip_distance,
        fare_amount,
        tip_amount
    FROM samples.nyctaxi.trips
    LIMIT 10
""")

df.show()

## 5. DataFrame Operations

In [ ]:
from pyspark.sql import functions as F

# Read the full table as a DataFrame
trips_df = spark.table("samples.nyctaxi.trips")

# Show schema
trips_df.printSchema()

In [ ]:
# Aggregate statistics
stats = trips_df.agg(
    F.count("*").alias("total_trips"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
    F.round(F.sum("fare_amount"), 2).alias("total_revenue")
)

stats.show()

In [ ]:
# Group by pickup hour
hourly_trips = (
    trips_df
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
    .groupBy("pickup_hour")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare")
    )
    .orderBy("pickup_hour")
)

hourly_trips.show(24)

## 6. Convert to Pandas

In [ ]:
# Convert Spark DataFrame to Pandas for local analysis
hourly_pandas = hourly_trips.toPandas()
hourly_pandas.head(10)

In [ ]:
# Plot with pandas (if matplotlib is installed)
try:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(hourly_pandas["pickup_hour"], hourly_pandas["trip_count"])
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Number of Trips")
    ax.set_title("NYC Taxi Trips by Hour")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib for plotting: uv pip install matplotlib")

## 7. Write Data (Example)

In [ ]:
# Example: Create a local DataFrame and write to Databricks
# Uncomment and modify catalog/schema to your own

# from datetime import datetime
# 
# data = [
#     (1, "Alice", datetime.now()),
#     (2, "Bob", datetime.now()),
#     (3, "Charlie", datetime.now())
# ]
# 
# df = spark.createDataFrame(data, ["id", "name", "created_at"])
# 
# df.write.mode("overwrite").saveAsTable("my_catalog.my_schema.test_table")

## 8. Clean Up

In [ ]:
# Stop the Spark session when done
# spark.stop()
# print("Spark session stopped.")

## 9. SaaS Monthly Sales Analysis

Analyzing sales trends from `sv_aibuilder_saas_demo.gold.fact_order` for the last 3 months.

In [7]:
# Monthly Sales Trend - Last 3 Months (Paying & Free Tier Users)
monthly_sales = spark.sql("""
    SELECT 
        DATE_FORMAT(order_month, 'yyyy-MM') AS month,
        COUNT(DISTINCT user_id) AS active_users,
        COUNT(*) AS total_orders,
        SUM(CASE WHEN is_revenue THEN order_value ELSE 0 END) AS revenue,
        ROUND(SUM(CASE WHEN is_revenue THEN order_value ELSE 0 END) / COUNT(DISTINCT user_id), 2) AS arpu
    FROM sv_aibuilder_saas_demo.gold.fact_order
    WHERE order_month >= ADD_MONTHS(DATE_TRUNC('month', CURRENT_DATE()), -3)
    GROUP BY order_month
    ORDER BY order_month DESC
""")

print("📊 Monthly Sales Trend (Last 3 Months)")
monthly_sales.show(truncate=False)

# Month-over-Month Growth
print("\n📈 Month-over-Month Changes")
spark.sql("""
    WITH monthly AS (
        SELECT 
            DATE_FORMAT(order_month, 'yyyy-MM') AS month,
            SUM(CASE WHEN is_revenue THEN order_value ELSE 0 END) AS revenue,
            COUNT(DISTINCT user_id) AS users
        FROM sv_aibuilder_saas_demo.gold.fact_order
        WHERE order_month >= ADD_MONTHS(DATE_TRUNC('month', CURRENT_DATE()), -3)
        GROUP BY order_month
        ORDER BY order_month
    )
    SELECT 
        month,
        revenue,
        users,
        ROUND(100 * (revenue - LAG(revenue) OVER (ORDER BY month)) / LAG(revenue) OVER (ORDER BY month), 1) AS revenue_growth_pct,
        ROUND(100 * (users - LAG(users) OVER (ORDER BY month)) / LAG(users) OVER (ORDER BY month), 1) AS user_growth_pct
    FROM monthly
""").show(truncate=False)

📊 Monthly Sales Trend (Last 3 Months)
+-------+------------+------------+----------+------+
|month  |active_users|total_orders|revenue   |arpu  |
+-------+------------+------------+----------+------+
|2026-02|5188        |7383        |2894496.61|557.92|
|2026-01|6359        |9965        |3822492.80|601.12|
|2025-12|6390        |10051       |3795013.58|593.90|
|2025-11|2296        |2601        |1000709.56|435.85|
+-------+------------+------------+----------+------+


📈 Month-over-Month Changes
+-------+----------+-----+------------------+---------------+
|month  |revenue   |users|revenue_growth_pct|user_growth_pct|
+-------+----------+-----+------------------+---------------+
|2025-11|1000709.56|2296 |NULL              |NULL           |
|2025-12|3795013.58|6390 |279.2             |178.3          |
|2026-01|3822492.80|6359 |0.7               |-0.5           |
|2026-02|2894496.61|5188 |-24.3             |-18.4          |
+-------+----------+-----+------------------+---------------+



## 10. Monthly Renewal Rate

**Calculation:** Users with revenue orders in both Month N-1 and Month N / Users in Month N-1

In [8]:
# Monthly Renewal Rate - Self-join to compare consecutive months
renewal_rate = spark.sql("""
    WITH monthly_users AS (
        SELECT DISTINCT 
            DATE_FORMAT(order_month, 'yyyy-MM') AS month,
            user_id
        FROM sv_aibuilder_saas_demo.gold.fact_order
        WHERE is_revenue = true
    ),
    retention AS (
        SELECT 
            curr.month,
            COUNT(DISTINCT curr.user_id) AS current_users,
            COUNT(DISTINCT prev.user_id) AS renewed_users,
            COUNT(DISTINCT CASE WHEN prev.user_id IS NULL THEN curr.user_id END) AS new_users
        FROM monthly_users curr
        LEFT JOIN monthly_users prev 
            ON curr.user_id = prev.user_id 
            AND prev.month = DATE_FORMAT(ADD_MONTHS(TO_DATE(CONCAT(curr.month, '-01')), -1), 'yyyy-MM')
        WHERE curr.month >= '2025-12'
        GROUP BY curr.month
    )
    SELECT 
        month,
        current_users,
        renewed_users,
        new_users,
        ROUND(100.0 * renewed_users / LAG(current_users) OVER (ORDER BY month), 1) AS renewal_rate_pct
    FROM retention
    ORDER BY month
""")

print("📊 Monthly Renewal Rate")
renewal_rate.show(truncate=False)

📊 Monthly Renewal Rate
+-------+-------------+-------------+---------+----------------+
|month  |current_users|renewed_users|new_users|renewal_rate_pct|
+-------+-------------+-------------+---------+----------------+
|2025-12|5289         |941          |4348     |NULL            |
|2026-01|5324         |2846         |2478     |53.8            |
|2026-02|4289         |2274         |2015     |42.7            |
+-------+-------------+-------------+---------+----------------+



## 11. Churn Risk Analysis

**Churn Signals (modify weights as needed):**
- `days_since_last_order` - Recency: >30 days = at risk
- `engagement_segment` - "dormant" or "casual" = higher risk
- `error_events` - Friction: high errors = frustrated user
- `total_orders` - Stickiness: <3 orders = not embedded
- `avg_order_value` trend - Declining spend = downgrade path

In [12]:
# Churn Risk Scoring - Rule-based approach (easy to modify)
# Each factor adds points: higher score = higher churn risk

churn_risk = spark.sql("""
    SELECT 
        user_id,
        company_name,
        subscription_tier,
        engagement_segment,
        DATEDIFF(CURRENT_DATE(), last_order_at) AS days_since_order,
        total_orders,
        ROUND(lifetime_revenue, 2) AS lifetime_revenue,
        error_events,
        
        -- RISK SCORING (adjust weights here)
        (CASE WHEN DATEDIFF(CURRENT_DATE(), last_order_at) > 45 THEN 30  -- Very stale
              WHEN DATEDIFF(CURRENT_DATE(), last_order_at) > 30 THEN 20  -- Stale
              WHEN DATEDIFF(CURRENT_DATE(), last_order_at) > 14 THEN 10  -- Cooling
              ELSE 0 END) +
        (CASE WHEN engagement_segment = 'dormant' THEN 25
              WHEN engagement_segment = 'casual' THEN 10 ELSE 0 END) +
        (CASE WHEN total_orders < 3 THEN 15 ELSE 0 END) +  -- Not sticky
        (CASE WHEN error_events > 5 THEN 10 ELSE 0 END)    -- High friction
        AS churn_risk_score,
        
        -- RISK TIER
        CASE 
            WHEN (CASE WHEN DATEDIFF(CURRENT_DATE(), last_order_at) > 45 THEN 30
                       WHEN DATEDIFF(CURRENT_DATE(), last_order_at) > 30 THEN 20
                       WHEN DATEDIFF(CURRENT_DATE(), last_order_at) > 14 THEN 10 ELSE 0 END +
                  CASE WHEN engagement_segment = 'dormant' THEN 25
                       WHEN engagement_segment = 'casual' THEN 10 ELSE 0 END +
                  CASE WHEN total_orders < 3 THEN 15 ELSE 0 END +
                  CASE WHEN error_events > 5 THEN 10 ELSE 0 END) >= 40 THEN 'HIGH'
            WHEN (CASE WHEN DATEDIFF(CURRENT_DATE(), last_order_at) > 45 THEN 30
                       WHEN DATEDIFF(CURRENT_DATE(), last_order_at) > 30 THEN 20
                       WHEN DATEDIFF(CURRENT_DATE(), last_order_at) > 14 THEN 10 ELSE 0 END +
                  CASE WHEN engagement_segment = 'dormant' THEN 25
                       WHEN engagement_segment = 'casual' THEN 10 ELSE 0 END +
                  CASE WHEN total_orders < 3 THEN 15 ELSE 0 END +
                  CASE WHEN error_events > 5 THEN 10 ELSE 0 END) >= 20 THEN 'MEDIUM'
            ELSE 'LOW'
        END AS risk_tier
        
    FROM sv_aibuilder_saas_demo.gold.agg_user_lifetime_summary
    WHERE last_order_at IS NOT NULL
""")

# Summary by risk tier
print("📊 Churn Risk Distribution")
churn_risk.groupBy("risk_tier").count().orderBy("risk_tier").show()

# Top at-risk accounts (by revenue)
print("⚠️ Top At-Risk Accounts (HIGH risk, sorted by revenue)")
from pyspark.sql.functions import desc
churn_risk.filter("risk_tier = 'HIGH'").orderBy(desc("lifetime_revenue")).show(10, truncate=False)

📊 Churn Risk Distribution
+---------+-----+
|risk_tier|count|
+---------+-----+
|     HIGH| 2201|
|      LOW| 4172|
|   MEDIUM| 3151|
+---------+-----+

⚠️ Top At-Risk Accounts (HIGH risk, sorted by revenue)
+----------+-----------------------------+-----------------+------------------+----------------+------------+----------------+------------+----------------+---------+
|user_id   |company_name                 |subscription_tier|engagement_segment|days_since_order|total_orders|lifetime_revenue|error_events|churn_risk_score|risk_tier|
+----------+-----------------------------+-----------------+------------------+----------------+------------+----------------+------------+----------------+---------+
|USR-008396|Google                       |free             |active            |63              |6           |4001.48         |7           |40              |HIGH     |
|USR-006172|Tupperware Brands Corporation|free             |casual            |62              |3           |2608.68        

In [13]:
# Save to sandbox for ML modeling or dashboards
churn_risk.write.mode("overwrite").saveAsTable("sv_aibuilder_saas_demo.saas_sandbox.churn_risk_scores")
print("✅ Saved to sv_aibuilder_saas_demo.saas_sandbox.churn_risk_scores")

# To upgrade to ML: Use this table as training data with Databricks AutoML
# spark.sql("SELECT * FROM sv_aibuilder_saas_demo.saas_sandbox.churn_risk_scores").display()

✅ Saved to sv_aibuilder_saas_demo.saas_sandbox.churn_risk_scores


## 12. Create ML Training Dataset

Build labeled dataset for churn prediction:
- **Observation point**: End of each month
- **Label**: Did user have a revenue order in the following month? (0=churned, 1=retained)
- **Features**: Behavior metrics as of observation date

In [14]:
# Create labeled training dataset for churn prediction
training_data = spark.sql("""
    WITH monthly_activity AS (
        -- Get each user's activity by month
        SELECT 
            user_id,
            DATE_TRUNC('month', order_month) AS obs_month,
            COUNT(*) AS orders_in_month,
            SUM(CASE WHEN is_revenue THEN order_value ELSE 0 END) AS revenue_in_month,
            MAX(CASE WHEN is_revenue THEN 1 ELSE 0 END) AS had_revenue_order
        FROM sv_aibuilder_saas_demo.gold.fact_order
        GROUP BY user_id, DATE_TRUNC('month', order_month)
    ),
    user_features AS (
        -- Get user-level features from lifetime summary
        SELECT 
            user_id,
            subscription_tier,
            account_age_days,
            total_orders,
            lifetime_revenue,
            avg_order_value,
            total_events,
            total_sessions,
            active_days,
            error_events,
            engagement_score,
            engagement_segment
        FROM sv_aibuilder_saas_demo.gold.agg_user_lifetime_summary
    ),
    labeled AS (
        -- Create label: did user have revenue in NEXT month?
        SELECT 
            curr.user_id,
            curr.obs_month,
            curr.orders_in_month,
            curr.revenue_in_month,
            -- Label: 1 = retained (had revenue next month), 0 = churned
            COALESCE(next.had_revenue_order, 0) AS retained,
            1 - COALESCE(next.had_revenue_order, 0) AS churned
        FROM monthly_activity curr
        LEFT JOIN monthly_activity next 
            ON curr.user_id = next.user_id 
            AND next.obs_month = ADD_MONTHS(curr.obs_month, 1)
        WHERE curr.obs_month < DATE_TRUNC('month', CURRENT_DATE())  -- Exclude current month (no label yet)
    )
    SELECT 
        l.user_id,
        l.obs_month,
        l.churned AS label,  -- Target variable for ML
        -- Features
        f.subscription_tier,
        f.account_age_days,
        l.orders_in_month,
        l.revenue_in_month,
        f.total_orders,
        f.lifetime_revenue,
        f.avg_order_value,
        f.total_events,
        f.total_sessions,
        f.active_days,
        f.error_events,
        f.engagement_score,
        f.engagement_segment
    FROM labeled l
    JOIN user_features f ON l.user_id = f.user_id
""")

# Show class balance
print("📊 Label Distribution (0=retained, 1=churned)")
training_data.groupBy("label").count().show()

# Show sample
print("📋 Sample Training Data")
training_data.show(5, truncate=False)

# Save for AutoML
training_data.write.mode("overwrite").saveAsTable("sv_aibuilder_saas_demo.saas_sandbox.churn_training_data")
print("✅ Saved to sv_aibuilder_saas_demo.saas_sandbox.churn_training_data")

📊 Label Distribution (0=retained, 1=churned)
+-----+-----+
|label|count|
+-----+-----+
|    0| 7357|
|    1| 7688|
+-----+-----+

📋 Sample Training Data
+----------+-------------------+-----+-----------------+----------------+---------------+----------------+------------+----------------+---------------+------------+--------------+-----------+------------+----------------+------------------+
|user_id   |obs_month          |label|subscription_tier|account_age_days|orders_in_month|revenue_in_month|total_orders|lifetime_revenue|avg_order_value|total_events|total_sessions|active_days|error_events|engagement_score|engagement_segment|
+----------+-------------------+-----+-----------------+----------------+---------------+----------------+------------+----------------+---------------+------------+--------------+-----------+------------+----------------+------------------+
|USR-009644|2025-12-01 00:00:00|0    |free             |240             |3              |816.83          |6           |34

## 13. Train with AutoML (Run in Databricks)

Copy this code to a **Databricks notebook** (AutoML requires cluster execution):

```python
from databricks import automl

# Run AutoML classification
summary = automl.classify(
    dataset=spark.table("sv_aibuilder_saas_demo.saas_sandbox.churn_training_data"),
    target_col="label",
    primary_metric="f1",           # Balance precision/recall
    timeout_minutes=30,            # Adjust based on urgency
    exclude_cols=["user_id", "obs_month"]  # Don't train on IDs
)

# Results
print(f"Best model: {summary.best_trial.model_description}")
print(f"F1 Score: {summary.best_trial.metrics['test_f1_score']:.3f}")

# View generated notebooks for feature importance & model details
summary.best_trial.notebook_url
```

**Alternative: Manual Model** (if you want more control)

## 14. AI-Powered Classification (No Training Required)

Use `ai_classify()` to leverage foundation models for instant churn risk classification. No model training needed - the LLM reasons about user behavior directly.

In [15]:
# AI-Powered Churn Classification using Foundation Models
# No training required - LLM reasons about user behavior

ai_predictions = spark.sql("""
    SELECT 
        user_id,
        company_name,
        subscription_tier,
        engagement_segment,
        days_since_order,
        lifetime_revenue,
        churn_risk_score,
        risk_tier AS rule_based_risk,
        
        -- AI Classification: LLM reasons about user context
        ai_classify(
            CONCAT(
                'SaaS user profile: ',
                'Subscription: ', subscription_tier, '. ',
                'Engagement level: ', engagement_segment, '. ',
                'Days since last order: ', days_since_order, '. ',
                'Lifetime revenue: $', ROUND(lifetime_revenue, 0), '. ',
                'Total orders: ', total_orders, '. ',
                'Error events: ', error_events, '.'
            ),
            ARRAY('high_churn_risk', 'medium_churn_risk', 'low_churn_risk')
        ) AS ai_predicted_risk
        
    FROM sv_aibuilder_saas_demo.saas_sandbox.churn_risk_scores
    WHERE days_since_order > 0
    LIMIT 20
""")

print("🤖 AI vs Rule-Based Churn Predictions")
ai_predictions.select(
    "company_name", "subscription_tier", "days_since_order", 
    "rule_based_risk", "ai_predicted_risk"
).show(20, truncate=False)

🤖 AI vs Rule-Based Churn Predictions
+-------------------------------+-----------------+----------------+---------------+-----------------+
|company_name                   |subscription_tier|days_since_order|rule_based_risk|ai_predicted_risk|
+-------------------------------+-----------------+----------------+---------------+-----------------+
|Popeyes Chicken & Biscuits     |professional     |31              |MEDIUM         |low_churn_risk   |
|Friendly Interior Design       |free             |12              |LOW            |low_churn_risk   |
|Reliable Guidance              |professional     |21              |LOW            |low_churn_risk   |
|Applied Industrial Technologies|free             |22              |MEDIUM         |medium_churn_risk|
|LeapFrog Enterprises           |enterprise       |4               |LOW            |low_churn_risk   |
|Gas Zone                       |free             |6               |MEDIUM         |medium_churn_risk|
|eBay                           |sta